# Practice 2

# 📓 Exercise 2 — Complete Standalone Notebook

# Exercise 2: Widening the GridSearchCV Parameter Space

**Goal:** Extend Section 10's `GridSearchCV` hyperparameter grid (`param_grid`) by adding one additional candidate value for each hyperparameter (`n_estimators` and `max_depth`). Evaluate whether the optimal parameters (`best_params_`) and the cross-validation score (`best_score_`) change with the wider search space, and analyze the increase in total computational fits.

# Cell 1 — Import required tools and modules

We import all necessary modules for data processing, train-test splitting, preprocessing pipelines, random forest regression, and grid search cross-validation.

In [9]:
# Import foundational libraries and sklearn components.

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor

# Cell 2 — Load dataset and engineer features

We load the California Housing dataset, create missing values for testing imputation, and compute feature interactions and categorical bins matching the standard workflow.

In [10]:
# Load dataset and prepare reproducible messy features.

california = fetch_california_housing(as_frame=True)
housing = california.frame

rng = np.random.default_rng(42)
messy = housing.copy()

# Introduce reproducible missing values

missing_idx = rng.choice(messy.index, size=200, replace=False)
messy.loc[missing_idx, "AveBedrms"] = np.nan
missing_idx_2 = rng.choice(messy.index, size=120, replace=False)
messy.loc[missing_idx_2, "AveRooms"] = np.nan

# Feature engineering

messy["coast_region"] = pd.cut(
    messy["Longitude"],
    bins=[-125, -121.22, -118.20, -114],
    labels=["Coastal", "Central", "Inland"],)
    
messy["income_cat"] = pd.cut(
    messy["MedInc"],
    bins=[0, 2.5, 4.5, 7, np.inf],
    labels=["Low", "Medium", "High", "Very High"],)

messy["bedrooms_ratio"] = messy["AveBedrms"] / messy["AveRooms"]
messy["rooms_per_person"] = messy["AveRooms"] / messy["AveOccup"]

X = messy.drop(columns="MedHouseVal")
y = messy["MedHouseVal"]

# Cell 3 — Split data into training and test sets

We partition the data into 80% training set and 20% test set using a fixed random seed.

In [11]:
# Perform reproducible train-test split.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Define feature subsets

numeric_features = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude"]

categorical_features = ["coast_region"]

# Cell 4 — Define the preprocessing pipeline

We assemble the ColumnTransformer to handle missing value imputation, standard scaling, and one-hot encoding automatically.

In [12]:
# Define sub-pipelines for numerical and categorical preprocessing.

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),])

# Combine into preprocessor ColumnTransformer
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),])

# Cell 5 — Construct the main estimation pipeline

We link the preprocessor and RandomForestRegressor together inside a single Pipeline object.

In [13]:
# Construct pipeline combining preprocessing and random forest regressor.

forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),])


# Cell 6 — Run baseline GridSearchCV from Section 10

We re-run Section 10's original 3x3 grid search space to establish baseline best parameters and score (9 combinations x 5 folds = 45 fits).

In [14]:
# Section 10 original param_grid: 3 values x 3 values = 9 combinations

param_grid_orig = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20],}

grid_search_orig = GridSearchCV(
    forest_pipeline,
    param_grid_orig,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1)

grid_search_orig.fit(X_train, y_train)

print("Original Best Params:", grid_search_orig.best_params_)
print(f"Original Best CV RMSE: {-grid_search_orig.best_score_:.4f}")

Original Best Params: {'model__max_depth': None, 'model__n_estimators': 300}
Original Best CV RMSE: 0.5091


# Cell 7 — Execute widened GridSearchCV 

We widen the search space by adding a 4th candidate value to each hyperparameter (`n_estimators: [100, 200, 300, 400]` and `max_depth: [None, 10, 20, 30]`), resulting in 16 combinations (80 fits total).

In [15]:
# widened param_grid: 4 values x 4 values = 16 combinations

param_grid_wide = {
    "model__n_estimators": [100, 200, 300, 400],
    "model__max_depth": [None, 10, 20, 30],}

grid_search_wide = GridSearchCV(
    forest_pipeline,
    param_grid_wide,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1)

grid_search_wide.fit(X_train, y_train)

print("Widened Best Params:", grid_search_wide.best_params_)
print(f"Widened Best CV RMSE: {-grid_search_wide.best_score_:.4f}")

Widened Best Params: {'model__max_depth': 30, 'model__n_estimators': 400}
Widened Best CV RMSE: 0.5082


# Cell 8 — Construct a summary comparison table

We compare the parameter search space, total number of fits, best hyperparameters, and best CV RMSE scores before and after widening the grid.

In [16]:
# Create DataFrame summarizing the effect of widening the grid search space.

comparison_data = {
    "Grid Search Version": ["Section 10 (Original)", "Exercise 2 (Widened)"],
    "Grid Dimensions": ["3 x 3 = 9 combinations", "4 x 4 = 16 combinations"],
    "Total Fits (5-fold CV)": [45, 80],
    "Best n_estimators": [
        grid_search_orig.best_params_["model__n_estimators"],
        grid_search_wide.best_params_["model__n_estimators"]],

    "Best max_depth": [
        str(grid_search_orig.best_params_["model__max_depth"]),
        str(grid_search_wide.best_params_["model__max_depth"])],

    "Best CV RMSE": [
        round(-grid_search_orig.best_score_, 4),
        round(-grid_search_wide.best_score_, 4)]}


comparison_df = pd.DataFrame(comparison_data)
comparison_df

,Grid Search Version,Grid Dimensions,Total Fits (5-fold CV),Best n_estimators,Best max_depth,Best CV RMSE
0,Section 10 (Original),3 x 3 = 9 combinations,45,300,None,0.5091
1,Exercise 2 (Widened),4 x 4 = 16 combinations,80,400,30,0.5082


# Cell 9 — Final interpretation and takeaways

We analyze whether best_params_ changed, evaluate the trade-off between computational cost and performance improvement, and draw conclusion.

In [17]:
# Report explicit findings answering Exercise 2 questions.

orig_params = grid_search_orig.best_params_
wide_params = grid_search_wide.best_params_

print("--- Exercise 2 Findings ---")
print(f"1. Did best_params_ change? {'Yes' if orig_params != wide_params else 'No'}")
print(f"   - Original: {orig_params}")
print(f"   - Widened:  {wide_params}")
print(f"2. Computational increase: Fits increased from 45 to 80 (+77.8%).")
print(f"3. Score change: CV RMSE improved from {-grid_search_orig.best_score_:.4f} to {-grid_search_wide.best_score_:.4f}.")

--- Exercise 2 Findings ---
1. Did best_params_ change? Yes
   - Original: {'model__max_depth': None, 'model__n_estimators': 300}
   - Widened:  {'model__max_depth': 30, 'model__n_estimators': 400}
2. Computational increase: Fits increased from 45 to 80 (+77.8%).
3. Score change: CV RMSE improved from 0.5091 to 0.5082.
